<a href="https://colab.research.google.com/github/kelll123/Tugas_ValiditasReliabilitas/blob/main/Tugas_ValiditasReliabilitas_Rachel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================
# LATIHAN DAN TUGAS
# UJI VALIDITAS DAN RELIABILITAS
# ============================================

import pandas as pd
import numpy as np
import re
from scipy import stats
from google.colab import files

# ============================================
# PERSIAPAN DATA
# ============================================

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

# Mengambil kolom pernyataan
kolom_butir = [
    c for c in df.columns
    if "[" in c and "]" in c
]

# Konversi skala Likert
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5
}

num = df[kolom_butir].replace(skala)
num = num.apply(pd.to_numeric, errors="coerce")

if num.isna().any().any():
    raise ValueError("Terdapat data kosong atau jawaban yang belum dikonversi.")

# Mengelompokkan berdasarkan dimensi
dimensi = {}

for kolom in kolom_butir:
    nama = re.match(r"^(.*?)\s*\[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

# Fungsi r tabel
def hitung_r_tabel(n):
    df = n - 2
    t = stats.t.ppf(0.975, df)
    return t / np.sqrt(df + t**2)

# Fungsi Cronbach's Alpha
def cronbach_alpha(data):
    k = data.shape[1]

    if k < 2:
        return np.nan

    var_item = data.var(axis=0, ddof=1).sum()
    var_total = data.sum(axis=1).var(ddof=1)

    if var_total == 0:
        return np.nan

    return (k / (k-1)) * (1 - var_item / var_total)

n = len(num)
r_tabel = hitung_r_tabel(n)

print("Jumlah responden:", n)
print("Jumlah dimensi:", len(dimensi))
print("Jumlah butir:", len(kolom_butir))
print("r tabel:", round(r_tabel, 3))


# ============================================
# LATIHAN 1
# INTERPRETASI VALIDITAS
# ============================================

print("\n========== LATIHAN 1 ==========")

hasil = []

for nama, daftar in dimensi.items():

    data = num[daftar]
    total = data.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):

        r_hitung, p = stats.pearsonr(
            data[kolom], total
        )

        corrected_r, _ = stats.pearsonr(
            data[kolom], total - data[kolom]
        )

        hasil.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": r_hitung,
            "r tabel": r_tabel,
            "Corrected r": corrected_r,
            "Keterangan": (
                "Valid" if r_hitung > r_tabel
                else "Tidak Valid"
            )
        })

validitas = pd.DataFrame(hasil)

display(validitas.round(3))

tidak_valid = validitas[
    validitas["Keterangan"] == "Tidak Valid"
]

print("\nButir tidak valid:", len(tidak_valid))

if len(tidak_valid) > 0:
    display(
        tidak_valid[
            ["Dimensi", "Butir", "r hitung", "r tabel"]
        ].round(3)
    )
else:
    print("Tidak terdapat butir yang tidak valid.")

print("""
Dugaan penyebab:
Butir yang tidak valid dapat disebabkan oleh
pernyataan yang kurang jelas, perbedaan pemahaman
responden, atau isi pernyataan yang kurang sesuai
dengan dimensi yang diukur.
""")


# ============================================
# LATIHAN 2
# PERBANDINGAN KRITERIA
# ============================================

print("\n========== LATIHAN 2 ==========")

jumlah_valid_r = (
    validitas["Keterangan"] == "Valid"
).sum()

jumlah_valid_corrected = (
    validitas["Corrected r"] >= 0.30
).sum()

print("Valid berdasarkan r hitung:", jumlah_valid_r)
print("Memenuhi Corrected r >= 0,30:", jumlah_valid_corrected)

print(
    "Selisih:",
    abs(jumlah_valid_r - jumlah_valid_corrected)
)

print("\nButir dengan Corrected r < 0,30:")

display(
    validitas[
        validitas["Corrected r"] < 0.30
    ][["Dimensi", "Butir", "Corrected r"]].round(3)
)

print("""
Perbedaan terjadi karena r hitung menggunakan
skor total yang masih mencakup butir yang diuji,
sedangkan Corrected r menggunakan skor total
setelah butir tersebut dikeluarkan.
""")


# ============================================
# LATIHAN 3
# PERBAIKAN INSTRUMEN
# ============================================

print("\n========== LATIHAN 3 ==========")

# Jika ada butir tidak valid, pilih salah satunya
# Jika tidak ada, gunakan Corrected r terendah
if len(tidak_valid) > 0:
    pilihan = tidak_valid.sort_values(
        "r hitung"
    ).iloc[0]
    alasan = "Butir tidak valid berdasarkan r hitung"
else:
    pilihan = validitas.sort_values(
        "Corrected r"
    ).iloc[0]
    alasan = "Evaluasi butir dengan Corrected r terendah"

nama_dimensi = pilihan["Dimensi"]
nama_butir = pilihan["Butir"]

daftar = dimensi[nama_dimensi]
nomor = int(nama_butir.replace("B", ""))
kolom_hapus = daftar[nomor - 1]

data_awal = num[daftar].copy()

alpha_sebelum = cronbach_alpha(data_awal)

data_baru = data_awal.drop(columns=[kolom_hapus])

alpha_sesudah = cronbach_alpha(data_baru)

print("Dimensi:", nama_dimensi)
print("Butir yang dipilih:", nama_butir)
print("Alasan:", alasan)

print("\nAlpha sebelum:", round(alpha_sebelum, 3))
print("Alpha sesudah:", round(alpha_sesudah, 3))
print("Perubahan alpha:", round(alpha_sesudah-alpha_sebelum, 3))

perbandingan_alpha = pd.DataFrame({
    "Keterangan": [
        "Sebelum penghapusan",
        "Sesudah penghapusan"
    ],
    "Cronbach's Alpha": [
        alpha_sebelum,
        alpha_sesudah
    ]
})

display(perbandingan_alpha.round(3))

if len(tidak_valid) == 0:
    print("""
CATATAN:
Tidak ditemukan butir tidak valid berdasarkan
r hitung. Penghapusan di atas merupakan simulasi
evaluasi berdasarkan Corrected r terendah,
bukan penghapusan butir tidak valid.
""")

# ============================================
# LATIHAN 4
# PENGARUH UKURAN SAMPEL
# ============================================

print("\n========== LATIHAN 4 ==========")

# Mengambil 20 responden secara acak
num_sample = num.sample(
    n=20,
    random_state=1
)

n_baru = len(num_sample)
r_tabel_baru = hitung_r_tabel(n_baru)

hasil_sample = []

for nama, daftar in dimensi.items():

    data = num_sample[daftar]
    total = data.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):

        r_hitung, p = stats.pearsonr(
            data[kolom], total
        )

        hasil_sample.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": r_hitung,
            "r tabel": r_tabel_baru,
            "Keterangan": (
                "Valid" if r_hitung > r_tabel_baru
                else "Tidak Valid"
            )
        })

validitas_sample = pd.DataFrame(hasil_sample)

display(validitas_sample.round(3))

valid_20 = (
    validitas_sample["Keterangan"] == "Valid"
).sum()

tidak_valid_20 = (
    validitas_sample["Keterangan"] == "Tidak Valid"
).sum()

perbandingan_sampel = pd.DataFrame({
    "Keterangan": [
        "Jumlah responden",
        "Derajat bebas",
        "r tabel",
        "Jumlah butir valid",
        "Jumlah butir tidak valid"
    ],
    "Sampel 38": [
        38, 36, round(r_tabel, 3),
        jumlah_valid_r,
        len(tidak_valid)
    ],
    "Sampel 20": [
        20, 18, round(r_tabel_baru, 3),
        valid_20,
        tidak_valid_20
    ]
})

print("\nPERBANDINGAN UKURAN SAMPEL")
display(perbandingan_sampel)


# ============================================
# LATIHAN 5
# REFLEKSI
# ============================================

print("\n========== LATIHAN 5 ==========")

print("""
Instrumen yang reliabel belum tentu valid karena
reliabilitas menunjukkan konsistensi hasil
pengukuran, sedangkan validitas menunjukkan
ketepatan instrumen dalam mengukur sesuatu
yang seharusnya diukur.

Contoh:
Timbangan yang selalu menunjukkan berat 5 kg
lebih tinggi dari berat sebenarnya merupakan
alat yang reliabel karena hasilnya konsisten.
Namun, alat tersebut tidak valid karena hasil
pengukurannya tidak sesuai dengan berat sebenarnya.
""")

print("\n========== SELESAI ==========")


Saving KUISIONER LATIHAN MK STATISTIKA PROBABILITAS (Responses) - Form Responses 1.csv to KUISIONER LATIHAN MK STATISTIKA PROBABILITAS (Responses) - Form Responses 1.csv
Jumlah responden: 38
Jumlah dimensi: 5
Jumlah butir: 27
r tabel: 0.32

========== LATIHAN 1 ==========


/tmp/ipykernel_490/1403453982.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = df[kolom_butir].replace(skala)


,Dimensi,Butir,r hitung,r tabel,Corrected r,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.723,0.32,0.547,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.393,0.32,0.260,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.428,0.32,0.257,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.470,0.32,0.240,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.703,0.32,0.523,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.739,0.32,0.627,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.738,0.32,0.550,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.651,0.32,0.516,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.750,0.32,0.567,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.643,0.32,0.430,Valid



Butir tidak valid: 0
Tidak terdapat butir yang tidak valid.

Dugaan penyebab:
Butir yang tidak valid dapat disebabkan oleh
pernyataan yang kurang jelas, perbedaan pemahaman
responden, atau isi pernyataan yang kurang sesuai
dengan dimensi yang diukur.


========== LATIHAN 2 ==========
Valid berdasarkan r hitung: 27
Memenuhi Corrected r >= 0,30: 23
Selisih: 4

Butir dengan Corrected r < 0,30:


,Dimensi,Butir,Corrected r
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.260
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.257
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.240
12,Aspek Psikologis penggunaan AI pada tugas atau...,B6,0.166



Perbedaan terjadi karena r hitung menggunakan
skor total yang masih mencakup butir yang diuji,
sedangkan Corrected r menggunakan skor total
setelah butir tersebut dikeluarkan.


========== LATIHAN 3 ==========
Dimensi: Aspek Psikologis penggunaan AI pada tugas atau projek kuliah
Butir yang dipilih: B6
Alasan: Evaluasi butir dengan Corrected r terendah

Alpha sebelum: 0.728
Alpha sesudah: 0.763
Perubahan alpha: 0.035


... (2 outputs left out)

,Dimensi,Butir,r hitung,r tabel,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.625,0.444,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.573,0.444,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.672,0.444,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.619,0.444,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.679,0.444,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.782,0.444,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.652,0.444,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.832,0.444,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.690,0.444,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.652,0.444,Valid



PERBANDINGAN UKURAN SAMPEL


,Keterangan,Sampel 38,Sampel 20
0,Jumlah responden,38.00,20.000
1,Derajat bebas,36.00,18.000
2,r tabel,0.32,0.444
3,Jumlah butir valid,27.00,27.000
4,Jumlah butir tidak valid,0.00,0.000



========== LATIHAN 5 ==========

Instrumen yang reliabel belum tentu valid karena
reliabilitas menunjukkan konsistensi hasil
pengukuran, sedangkan validitas menunjukkan
ketepatan instrumen dalam mengukur sesuatu
yang seharusnya diukur.

Contoh:
Timbangan yang selalu menunjukkan berat 5 kg
lebih tinggi dari berat sebenarnya merupakan
alat yang reliabel karena hasilnya konsisten.
Namun, alat tersebut tidak valid karena hasil
pengukurannya tidak sesuai dengan berat sebenarnya.


========== SELESAI ==========
